In [ ]:
<div style="background-color: #e0f7ff;  
            border-radius: 20px;  
            padding: 25px;  
            text-align: center;    
            font-family: 'Courier New', monospace;">
  <h1 style="font-size: 42px;">CMG Data Ingestion</h1>
  <p style="font-size: 16px; margin-top: 18px;">
    This notebook demonstrates how to parse and convert the CMG dataset, containing blood glucose and insulin data 
    from a person with type 1 diabetes, into structured CSV files for easy analysis and exploration.
    We will parse the raw text file, explore the general structure of the dataset, and prepare it for further analysis.
  </p>
  <h2 style="margin-top: 25px; font-size: 24px;">This Notebook Covers:</h2>
  <ul style="text-align: left; display: inline-block; font-size: 15px; margin-top: 8px;">
    <li>General information about the dataset</li>
    <li>Parsing and converting raw text data into CSV files</li>
    <li>Basic exploration of data sections (Meal, Insulin, Glucose, etc.)</li>
    <li>Loading parsed CSV files into a database (CSV → SQL)</li>
  </ul>
</div>


<div style="background-color: #e0f7ff;  
            border-radius: 20px;  
            padding: 30px;  
            font-family: 'Courier New', monospace;  
            line-height: 1.5;">

  <h2 style="text-align: center; font-size: 32px;">General Information about the Dataset</h2>

  <ul style="font-size: 16px; margin-top: 20px;">
    <li><strong>Source:</strong> Personal CMG measurements from a person with type 1 diabetes.</li>
    <li><strong>Content:</strong> The dataset contains multiple sections:
      <ul>
        <li><code>Meal</code> – meal times and carbohydrate intake</li>
        <li><code>Insulin_bolus</code> – bolus insulin injections</li>
        <li><code>Insulin_infusion</code> – continuous insulin infusion rates</li>
        <li><code>Glucose_concentration</code> – sensor glucose readings</li>
        <li><code>Fingerstick_glucose_concentration</code> – fingerstick measurements</li>
        <li>Events: <code>Priming_event</code>, <code>Refill_event</code>, <code>Sensor_inserted</code>, <code>Sensor_stopped</code>, <code>Audio_alerts</code>, <code>Vibrate_alerts</code></li>
      </ul>
    </li>
    <li><strong>Format:</strong> Raw <code>.txt</code> file, tab-delimited</li>
    <li><strong>Purpose:</strong> Parsing and conversion to CSV for further exploration and analysis</li>
  </ul>
</div>


<div style="background-color: #e0f7ff;  
            border-radius: 20px;  
            padding: 30px;  
            text-align: center;    
            font-family: 'Courier New', monospace;  
            line-height: 1.5; margin-top: 20px;">

  <h3 style="font-size: 28px;">Sample of the raw CMG data</h3>

  <pre style="font-family: 'Courier New', monospace; text-align: left; display: inline-block; margin-top: 20px; overflow-x: auto;">
Meal ************************************************************
Time                        CHO
(dd/mm/yyyy hh:mm)          (g)
08/07/2025 08:56	35
08/07/2025 10:57	10
08/07/2025 12:02	5

Glucose_concentration ********************************************
Time                        conc
(dd/mm/yyyy hh:mm)         (mmol/L)
08/07/2025 00:02	8.27132306
08/07/2025 00:07	8.21581306
08/07/2025 14:25	25
  </pre>

</div>

<div style="background-color: #e0f7ff;  
            border-radius: 20px;  
            padding: 30px;  
            font-family: 'Courier New', monospace;  
            line-height: 1.5;">

  <h2 style="text-align: center; font-size: 28px;">Parsing and converting raw text data into CSV files</h2>
</div>

<div style="background-color: #e0f7ff;  
            border-radius: 20px;  
            padding: 30px;     
            font-family: 'Courier New', monospace;  
            line-height: 1.5; margin-top: 20px;">

  <h2 style="font-size: 28px;">Step 1: Read the raw data</h2>

  <p style="font-size: 16px; margin-top: 20px;">
    We first import pandas and read the raw <code>.txt</code> file into Python. Each line will later be assigned to a section for parsing and conversion.
  </p>

</div>


In [9]:
import pandas as pd

raw_data_path = '../data/raw/'
input_txt_file = 'cmg-data.txt'

with open(f'{raw_data_path}{input_txt_file}', 'r') as f:
    lines = f.readlines()


<div style="background-color: #e0f7ff;  
            border-radius: 20px;  
            padding: 30px;   
            font-family: 'Courier New', monospace;  
            line-height: 1.5; margin-top: 20px;">

  <h2 style="font-size: 28px;">Step 2: Split data into sections</h2>

  <p style="font-size: 16px; margin-top: 20px;">
    Each section of the dataset (Meal, Insulin, Glucose, etc.) is separated and stored in a dictionary. 
    This will allow us to process each section individually and convert them into CSV files.
  </p>

</div>


In [12]:
data = {
    "Meal": [], "Insulin_bolus": [], "Insulin_infusion": [],
    "Glucose_concentration": [], "Fingerstick_glucose_concentration": [],
    "Priming_event": [], "Refill_event": [], "Sensor_inserted": [],
    "Sensor_stopped": [], "Audio_alerts": [], "Vibrate_alerts": []
}

current_section = None
for line in lines:
    line = line.strip()
    if not line:
        continue
    elif 'Meal' in line:
        current_section = 'Meal'
    elif 'Insulin_bolus' in line:
        current_section = 'Insulin_bolus'
    elif 'Insulin_infusion' in line:
        current_section = 'Insulin_infusion'
    elif 'Glucose_concentration' in line:
        current_section = 'Glucose_concentration'
    elif 'Fingerstick_glucose_concentration' in line:
        current_section = 'Fingerstick_glucose_concentration'
    elif 'Priming_event' in line:   
        current_section = 'Priming_event'
    elif 'Refill_event' in line:
        current_section = 'Refill_event'
    elif 'Sensor_inserted' in line:
        current_section = 'Sensor_inserted'
    elif 'Sensor_stopped' in line:
        current_section = 'Sensor_stopped'
    elif 'Audio_alerts' in line:
        current_section = 'Audio_alerts'
    elif 'Vibrate_alerts' in line:
        current_section = 'Vibrate_alerts'
    else:
        if current_section:
            data[current_section].append(line)

# Remove first two lines (headers)
data_without_headers = {k: v[2:] for k, v in data.items()}


<div style="background-color: #e0f7ff;  
            border-radius: 20px;  
            padding: 30px;    
            font-family: 'Courier New', monospace;  
            line-height: 1.5; margin-top: 20px;">

  <h2 style="font-size: 28px;">Step 3: Convert each section to CSV</h2>

  <p style="font-size: 16px; margin-top: 20px;">
    Each section is converted to a Pandas DataFrame with proper data types and saved as a separate CSV file for analysis. 
    This ensures that all measurements are properly formatted and ready for further exploration.
  </p>

</div>


In [15]:
processed_data_path = '../data/processed/'

headers = {
    'Meal': ['Time', 'CHO'],
    'Insulin_bolus': ['Time', 'Bolus', 'Duration'],
    'Insulin_infusion': ['Time', 'Rate'],
    'Glucose_concentration': ['Time', 'Conc'],
    'Fingerstick_glucose_concentration': ['Time', 'Conc'],
    'Priming_event': ['Time'],
    'Refill_event': ['Time'],
    'Sensor_inserted': ['Time'],
    'Sensor_stopped': ['Time'],
    'Audio_alerts': ['Time'],
    'Vibrate_alerts': ['Time']
}

for section_name, lines in data_without_headers.items():
    rows = [line.split('\t') for line in lines]
    df = pd.DataFrame(rows, columns=headers[section_name])
    
    if 'Conc' in df.columns:
        df['Conc'] = df['Conc'].str.replace(r'[^0-9.]', '', regex=True).astype(float)
    if 'Time' in df.columns:
        df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%Y %H:%M')
    
    df.to_csv(f'{processed_data_path}{section_name}.csv', index=False)


In [ ]:
<div style="background-color: #e0f7ff;  
            border-radius: 20px;  
            padding: 30px;  
            text-align: center;    
            font-family: 'Courier New', monospace;  
            line-height: 1.5; margin-top: 20px;">

  <h2 style="font-size: 32px;">Basic exploration of data sections</h2>
</div>


In [18]:
meal_df = pd.read_csv('../data/processed/Meal.csv')
meal_df.head()


,Time,CHO
0,2025-07-08 08:56:00,35
1,2025-07-08 10:57:00,10
2,2025-07-08 12:02:00,5
3,2025-07-08 14:25:00,25
4,2025-07-08 15:36:00,15


In [19]:
insulin_bolus_df = pd.read_csv('../data/processed/Insulin_bolus.csv')
insulin_bolus_df.head()

,Time,Bolus,Duration
0,2025-07-08 08:56:00,5.0,-
1,2025-07-08 10:57:00,1.0,-
2,2025-07-08 12:02:00,0.5,-
3,2025-07-08 14:25:00,2.5,-
4,2025-07-08 15:36:00,1.5,-


<div style="background-color: #e0f7ff;  
            border-radius: 20px;  
            padding: 30px;  
            text-align: center;    
            font-family: 'Courier New', monospace;  
            line-height: 1.5;  
            margin-top: 20px;">

  <h2 style="font-size: 32px;">Loading CSV Files into Database (CSV → Sqlite)</h2>
</div>


In [10]:
import sqlite3
import pandas as pd

DATA_PROCESSED = '../data/processed'

sections = [
    "Meal",
    "Insulin_bolus",
    "Insulin_infusion",
    "Glucose_concentration",
    "Fingerstick_glucose_concentration",
    "Priming_event",
    "Refill_event",
    "Sensor_inserted",
    "Sensor_stopped",
    "Audio_alerts",
    "Vibrate_alerts"
]

conn = sqlite3.connect(f'{DATA_PROCESSED}/cgm-db')
cur = conn.cursor()

for i in sections:
    df = pd.read_csv(f'{DATA_PROCESSED}/{i}.csv')
    df.to_sql(name=i.lower(), if_exists='replace', con=conn)

df_meal = pd.read_sql("SELECT * FROM meal LIMIT 5;", conn)
print(df_meal)

   index                 Time  CHO
0      0  2025-07-08 08:56:00   35
1      1  2025-07-08 10:57:00   10
2      2  2025-07-08 12:02:00    5
3      3  2025-07-08 14:25:00   25
4      4  2025-07-08 15:36:00   15
